In [ ]:
pip install numpy gplearn sympy pysindy pandas scikit-learn

### *Abordagem Inicial*

> Tentar calcular uma previsão de uma função simples (limitada a grau exponencial 1): q' = arrival_rate + throughput_rate.

- Arrival_rate: Quantos pacotes chegam num estante de tempo;
- throughput_rate: Quantos pacotes saem num estante de tempo.

*Justificativa de "throughput_rate"*: Ao usar outra variável como "departure_rate" a capacidade do modelo se mostrou inferior ao usar a variável escolhida.

**Ideia:** Gerar um modelo que consiga fazer um previsão utilizando derivação + método de Euler para prever o estado da fila em um estante de tempo com base na variação da fila do instante de tempo anterior.

**Problema da abordagem:** Introduz muita propagação de erro: Erro da derivação + erro de Euler. A tornando difícil ser usada em modelos com rollout (janela de previsão maior que um passo - One Step).

In [ ]:
import numpy as np
import pandas as pd
import pysindy as ps
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Dados e fatiamento
url = 'https://raw.githubusercontent.com/Surtivo/SR_PySINDy/refs/heads/main/ns3_simulation_queue_metrics.csv'
df = pd.read_csv(url)

n = len(df)
n_train = int(n * 0.8)
n_test = int(n * 0.1)

df_train = df.iloc[:n_train].copy()
df_test = df.iloc[n_train:n_train+n_test].copy()
df_val = df.iloc[n_train+n_test:].copy()

# Separando u_in (arrival) e u_out (throughput)
def get_arrays(data):
    return (
        data["time"].values,
        data[["queue_packets"]].values,
        data[["arrival_rate_mbps", "throughput_mbps"]].values
    )

t_train, X_train, U_train = get_arrays(df_train)
t_test, X_test, U_test = get_arrays(df_test)
t_val, X_val, U_val = get_arrays(df_val)

# 2. Treino (80%) com 2 sinais de controle
model = ps.SINDy(
    differentiation_method=ps.SmoothedFiniteDifference(),
    feature_library=ps.PolynomialLibrary(degree=1, include_bias=False),
    optimizer=ps.STLSQ(threshold=0.01)
)

model.fit(X_train, t=t_train, u=U_train, feature_names=["q", "u_in", "u_out"])
print("--- Equação da Fila (Controles Separados) ---")
model.print()

# 3. Avaliação "One-Step-Ahead"
def evaluate_one_step(X_block, U_block, dt=0.1, block_name="Bloco"):
    dq_dt_pred = model.predict(X_block, u=U_block)[:, 0]

    q_pred = X_block[:-1, 0] + dq_dt_pred[:-1] * dt
    q_true = X_block[1:, 0]

    r2 = r2_score(q_true, q_pred)
    rmse = np.sqrt(mean_squared_error(q_true, q_pred))
    mae = mean_absolute_error(q_true, q_pred)

    print(f"\n--- Métricas One-Step: {block_name} ---")
    print(f"R² Score: {r2:.4f}")
    print(f"RMSE:     {rmse:.4f} pacotes")
    print(f"MAE:      {mae:.4f} pacotes")

evaluate_one_step(X_test, U_test, dt=0.1, block_name="Teste (10%)")
evaluate_one_step(X_val, U_val, dt=0.1, block_name="Validação Final (10%)")

##### Saída:
```
--- Equação da Fila (Controles Separados) ---
(q)' = -0.718 q +  1.016 u_in + -0.057 u_out

--- Métricas One-Step: Teste (10%) ---
R² Score: 0.8680
RMSE:     2.0411 pacotes
MAE:      1.8248 pacotes

--- Métricas One-Step: Validação Final (10%) ---
R² Score: 0.8452
RMSE:     1.5054 pacotes
MAE:      1.3785 pacotes

##### Mudança para contemplar tudo em pacotes. Grau alterado para 3

In [ ]:
import numpy as np
import pandas as pd
import pysindy as ps
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Dados e fatiamento
url = 'https://raw.githubusercontent.com/Surtivo/SR_PySINDy/refs/heads/main/ns3_simulation_queue_metrics.csv'
df = pd.read_csv(url)
L = 1500  # Tamanho do pacote em bytes
df["net_arrival_pps"] = (df["arrival_rate_mbps"] * 10**6) / (8 * L)
df["throughput_pps"] = (df["throughput_mbps"] * 10**6) / (8 * L)

n = len(df)
n_train = int(n * 0.8)
n_test = int(n * 0.1)

df_train = df.iloc[:n_train].copy()
df_test = df.iloc[n_train:n_train+n_test].copy()
df_val = df.iloc[n_train+n_test:].copy()

def get_arrays(data):
    return data["time"].values, data[["queue_packets"]].values, data[["net_arrival_pps"]].values, data[["throughput_pps"]].values

t_train, X_train, A_train, T_train = get_arrays(df_train)
t_test, X_test, A_teste, T_test = get_arrays(df_test)
t_val, X_val, A_val, T_val = get_arrays(df_val)

train_stack = np.column_stack((A_train, T_train))
test_stack = np.column_stack((A_teste, T_test))
val_stack = np.column_stack((A_val, T_val))

# 2. Treino (80%)
model = ps.SINDy(
    differentiation_method=ps.SmoothedFiniteDifference(),
    feature_library=ps.PolynomialLibrary(degree=3, include_bias=False),
    optimizer=ps.STLSQ(threshold=0.0001)
)
model.fit(X_train, t=t_train, u=train_stack, feature_names=["q", "net_arrival_pps", "throughput_pps"])
print("--- Equação da Fila ---")
model.print()

# 3. Avaliação "One-Step-Ahead" (Previsão de 1 Passo à Frente)
def evaluate_one_step(X_block, U_block, dt=0.1, block_name="Bloco"):
    # Calcula a derivada dq/dt prevista pelo SINDy em cada instante t
    dq_dt_pred = model.predict(X_block, u=U_block)[:, 0]

    # Previsão do próximo estado: q(t + dt) = q(t) + dq/dt * dt
    q_pred = X_block[:-1, 0] + dq_dt_pred[:-1] * dt
    q_true = X_block[1:, 0]  # Valor real medido no passo seguinte

    # Métricas
    r2 = r2_score(q_true, q_pred)
    rmse = np.sqrt(mean_squared_error(q_true, q_pred))
    mae = mean_absolute_error(q_true, q_pred)

    print(f"\n--- Métricas One-Step: {block_name} ---")
    print(f"R² Score: {r2:.4f}")
    print(f"RMSE:     {rmse:.4f} pacotes")
    print(f"MAE:      {mae:.4f} pacotes")

evaluate_one_step(X_test, test_stack, dt=0.1, block_name="Teste (10%)")
evaluate_one_step(X_val, val_stack, dt=0.1, block_name="Validação Final (10%)")

##### Saída:

```
--- Equação da Fila ---
(q)' = -14.170 q +  0.145 net_arrival_pps + -0.145 throughput_pps +  0.105 q^2 + -0.006 q net_arrival_pps +  0.028 q throughput_pps +  0.012 q^3 + -0.001 q^2 throughput_pps

--- Métricas One-Step: Teste (10%) ---
R² Score: 0.9439
RMSE:     1.3302 pacotes
MAE:      1.1977 pacotes

--- Métricas One-Step: Validação Final (10%) ---
R² Score: 0.9787
RMSE:     0.5582 pacotes
MAE:      0.3410 pacotes

#### Utilização de métricas como estado de fila (se está cheia ou não) e capacidade máxima da fila
> Ampliar as variáveis de controle do modelo para contemplar essas variávais pode ajudar a montar uma equação de previsão melhor

In [ ]:
# Código original perdido. Reaproveitado da comparação entre modelos

import numpy as np
import pandas as pd
import pysindy as ps
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Carregamento e Preparação Inicial
url = 'https://raw.githubusercontent.com/Surtivo/SR_PySINDy/refs/heads/main/ns3_simulation_queue_metrics.csv'
df_raw = pd.read_csv(url)

L = 1500  # MTU em bytes
horizontes = [1, 3, 5, 10]

# Função de Rollout Reutilizável
def evaluate_rollout(model, X_block, df_block, steps=1, dt=0.1, use_queue_full=False, enforce_queue_cap=False, q_max=20.0):
    U_base = df_block[["net_arrival_pps", "throughput_pps"]].values
    num_samples = len(X_block) - steps
    q_preds, q_trues = [], []

    for i in range(num_samples):
        q_sim = X_block[i, 0]
        for s in range(steps):
            u_in_out = U_base[i + s]
            if use_queue_full:
                q_full_val = 1.0 if q_sim >= q_max else 0.0
                u_step = np.append(u_in_out, q_full_val)
            else:
                u_step = u_in_out

            dq_dt = model.predict(np.array([[q_sim]]), u=np.array([u_step]))[0, 0]
            q_sim = q_sim + dq_dt * dt

            if enforce_queue_cap:
                q_sim = np.clip(q_sim, 0.0, q_max)
            else:
                q_sim = max(0.0, q_sim)

        q_preds.append(q_sim)
        q_trues.append(X_block[i + steps, 0])

    r2 = r2_score(q_trues, q_preds)
    rmse = np.sqrt(mean_squared_error(q_trues, q_preds))
    mae = mean_absolute_error(q_trues, q_preds)
    return r2, rmse, mae

# Função para Executar o Grid de Modelos
def run_experiments(df_input, target_saida_col):
    df = df_input.copy()
    df["net_arrival_pps"] = (df["arrival_rate_mbps"] * 10**6) / (8 * L)
    df["throughput_pps"] = (df[target_saida_col] * 10**6) / (8 * L)
    df["queue_full"] = (df["queue_packets"] >= 20).astype(float)

    n_train = int(len(df) * 0.8)
    df_train = df.iloc[:n_train].copy()
    df_val = df.iloc[n_train:].copy()

    t_train = df_train["time"].values
    X_train = df_train[["queue_packets"]].values
    X_val = df_val[["queue_packets"]].values

    results = []

    for grau in range(1, 4):
        for use_qfull in [False, True]:
            for use_cap in [False, True]:
                if use_qfull:
                    cols_u = ["net_arrival_pps", "throughput_pps", "queue_full"]
                    f_names = ["q", "net_arrival_pps", "throughput_pps", "queue_full"]
                else:
                    cols_u = ["net_arrival_pps", "throughput_pps"]
                    f_names = ["q", "net_arrival_pps", "throughput_pps"]

                U_train = df_train[cols_u].values

                feature_lib = ps.PolynomialLibrary(degree=grau, include_bias=False)
                model = ps.SINDy(
                    differentiation_method=ps.SmoothedFiniteDifference(),
                    feature_library=feature_lib,
                    optimizer=ps.STLSQ(threshold=0.0001)
                ).fit(X_train, t=t_train, u=U_train, feature_names=f_names)

                coefs = model.coefficients()[0]
                n_terms = np.sum(coefs != 0)

                row = {
                    "Grau": grau,
                    "Q_Full": "Sim" if use_qfull else "Não",
                    "Cap": "Sim" if use_cap else "Não",
                    "Termos": n_terms
                }

                for h in horizontes:
                    r2, rmse, mae = evaluate_rollout(
                        model, X_val, df_val, steps=h,
                        use_queue_full=use_qfull, enforce_queue_cap=use_cap
                    )
                    row[f"R² ({h}s)"] = r2
                    row[f"RMSE ({h}s)"] = rmse
                    row[f"MAE ({h}s)"] = mae

                results.append(row)
        # print("========================== Equação Grau:", grau, "- Alvo:", target_saida_col, "============================",)
        # model.print()
        # print("==========================================================================================\n")

    return pd.DataFrame(results)

df_res_throughput = run_experiments(df_raw, "throughput_mbps")

print(df_res_throughput.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

##### Saída:

```
Grau Q_Full Cap  Termos  R² (1s)  RMSE (1s)  MAE (1s)  R² (3s)  RMSE (3s)  MAE (3s)  R² (5s)  RMSE (5s)  MAE (5s)  R² (10s)  RMSE (10s)  MAE (10s)
    1    Não Não       3   0.9239     1.7693    1.5887   0.2003     4.7122    4.2996  -1.8335     7.0134    6.3755   -7.5100     10.7266     9.5752
    1    Não Sim       3   0.9239     1.7693    1.5887   0.2003     4.7122    4.2996  -1.8335     7.0134    6.3755   -7.5100     10.7266     9.5752
    1    Sim Não       4   0.9226     1.7841    1.5983   0.1657     4.8128    4.3795  -2.0129     7.2320    6.5512   -8.3999     11.2736    10.0354
    1    Sim Sim       4   0.9226     1.7841    1.5983   0.1657     4.8128    4.3795  -2.0129     7.2320    6.5512   -8.3999     11.2736    10.0354
    2    Não Não       6   0.9738     1.0378    0.8657   0.7673     2.5419    2.2623   0.3326     3.4037    3.0494   -0.2361      4.0882     3.7261
    2    Não Sim       6   0.9748     1.0189    0.8490   0.7741     2.5047    2.2395   0.3471     3.3667    3.0270   -0.2255      4.0706     3.7140
    2    Sim Não      11   0.9819     0.8632    0.6547   0.8303     2.1709    1.8176   0.3770     3.2887    2.7043   -0.2060      4.0381     3.4770
    2    Sim Sim      11   0.9825     0.8476    0.6427   0.8431     2.0870    1.7767   0.4774     3.0119    2.5823   -0.0775      3.8169     3.3511
    3    Não Não       8   0.9764     0.9849    0.7387   0.7706     2.5236    2.1309   0.2056     3.7135    3.2551   -1.3412      5.6263     5.3330
    3    Não Sim       8   0.9772     0.9690    0.7251   0.7800     2.4716    2.1005   0.2444     3.6218    3.2054   -1.2936      5.5687     5.2867
    3    Sim Não      25   0.9792     0.9251    0.7289   0.8003     2.3545    1.9741   0.0756     4.0059    3.1841   -4.2805      8.4496     6.2426
    3    Sim Sim      25   0.9792     0.9243    0.7281   0.8090     2.3026    1.9457   0.2307     3.6544    3.0374   -3.1276      7.4705     5.8721

### Utilização de abordagem discreta
> Vantagem: Reduz a propagação de erros ao eliminar etapas como derivação e método de Euler.

- Essa abordagem foi consideravelmente melhor ao aplicar rollout, pois foi capaz de propagar menos erros ao tentar prever uma equação a variação de estado da fila ao invés de tentar prever uma taxa de variação da fila e usar essa função para estimar o próximo estado da fila.

In [ ]:
import numpy as np
import pandas as pd
import pysindy as ps

# Dados
url = "https://raw.githubusercontent.com/Surtivo/SR_PySINDy/refs/heads/main/ns3_simulation_queue_metrics.csv"
df = pd.read_csv(url)

L = 1500  # bytes

df["arrival_pps"] = (
    df["arrival_rate_mbps"] * 1e6 / (8 * L)
)

df["departure_pps"] = (df["throughput_mbps"] * 10**6) / (8 * L)

df["queue_full"] = (
    df["queue_packets"] >= 20
).astype(float)

n = len(df)
n_train = int(0.8 * n)

df_train = df.iloc[:n_train].copy()
df_val = df.iloc[n_train:].copy()

X_train = df_train[["queue_packets"]].values
X_val = df_val[["queue_packets"]].values

U_train = df_train[
    ["arrival_pps", "departure_pps"]
].values

U_val = df_val[
    ["arrival_pps", "departure_pps"]
].values

feature_lib = ps.PolynomialLibrary(
    degree=1,
    include_bias=False
)

optimizer = ps.STLSQ(
    threshold=0.0001
)

model = ps.DiscreteSINDy(
    feature_library=feature_lib,
    optimizer=optimizer
)

model.fit(
    X_train,
    t=0.1,
    u=U_train,
    feature_names=[
        "q",
        "arrival_pps",
        "departure_pps"
    ]
)

model.print()

##### Saída:
```
(q)[k+1] =  0.887 q[k] +  0.001 arrival_pps[k] +  0.001 departure_pps[k]

In [ ]:
import numpy as np
import pandas as pd
import pysindy as ps
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Carregamento e Preparação Inicial
url = 'https://raw.githubusercontent.com/Surtivo/SR_PySINDy/refs/heads/main/ns3_simulation_queue_metrics.csv'
df_raw = pd.read_csv(url)

L = 1500  # MTU em bytes
horizontes = [1, 3, 5, 10]

# Função de Rollout Autoregressivo Discreto
def evaluate_discrete_rollout(model, X_block, df_block, steps=1, use_queue_full=False, enforce_queue_cap=False, q_max=20.0):
    U_base = df_block[["net_arrival_pps", "throughput_pps"]].values
    num_samples = len(X_block) - steps
    q_preds, q_trues = [], []

    for i in range(num_samples):
        q_sim = X_block[i, 0]

        # Iteração por passos discretos: q[k+1] = f(q[k], u[k])
        for s in range(steps):
            u_in_out = U_base[i + s]
            if use_queue_full:
                q_full_val = 1.0 if q_sim >= q_max else 0.0
                u_step = np.append(u_in_out, q_full_val)
            else:
                u_step = u_in_out

            # Previsão direta do próximo estado q[k+1]
            q_next = model.predict(np.array([[q_sim]]), u=np.array([u_step]))[0, 0]

            # Restrições físicas
            if enforce_queue_cap:
                q_sim = np.clip(q_next, 0.0, q_max)
            else:
                q_sim = max(0.0, q_next)

        q_preds.append(q_sim)
        q_trues.append(X_block[i + steps, 0])

    r2 = r2_score(q_trues, q_preds)
    rmse = np.sqrt(mean_squared_error(q_trues, q_preds))
    mae = mean_absolute_error(q_trues, q_preds)
    return r2, rmse, mae

# Função para Executar a Varredura Discreta
def run_discrete_experiments(df_input, target_saida_col):
    df = df_input.copy()
    df["net_arrival_pps"] = (df["arrival_rate_mbps"] * 10**6) / (8 * L)
    df["throughput_pps"] = (df[target_saida_col] * 10**6) / (8 * L)
    df["queue_full"] = (df["queue_packets"] >= 20).astype(float)

    n_train = int(len(df) * 0.8)
    df_train = df.iloc[:n_train].copy()
    df_val = df.iloc[n_train:].copy()

    # Prepara os pares de transição de tempo discreto: x[k] -> x[k+1]
    X_train_k = df_train[["queue_packets"]].values[:-1]
    X_train_k1 = df_train[["queue_packets"]].values[1:]
    t_train_k = df_train["time"].values[:-1]

    X_val = df_val[["queue_packets"]].values

    results = []

    for grau in range(1, 4):
        for use_qfull in [False, True]:
            for use_cap in [False, True]:
                if use_qfull:
                    cols_u = ["net_arrival_pps", "throughput_pps", "queue_full"]
                    f_names = ["q", "net_arrival_pps", "throughput_pps", "queue_full"]
                else:
                    cols_u = ["net_arrival_pps", "throughput_pps"]
                    f_names = ["q", "net_arrival_pps", "throughput_pps"]

                U_train_k = df_train[cols_u].values[:-1]

                feature_lib = ps.PolynomialLibrary(degree=grau, include_bias=False)

                model = ps.SINDy(
                    differentiation_method=None,
                    feature_library=feature_lib,
                    optimizer=ps.STLSQ(threshold=0.0001)
                )

                # Passa o 't=t_train_k' obrigatório e sobrescreve o alvo com 'x_dot=X_train_k1'
                model.fit(X_train_k, t=t_train_k, x_dot=X_train_k1, u=U_train_k, feature_names=f_names)

                coefs = model.coefficients()[0]
                n_terms = np.sum(coefs != 0)

                row = {
                    "Grau": grau,
                    "Q_Full": "Sim" if use_qfull else "Não",
                    "Cap": "Sim" if use_cap else "Não",
                    "Termos": n_terms
                }

                for h in horizontes:
                    r2, rmse, mae = evaluate_discrete_rollout(
                        model, X_val, df_val, steps=h,
                        use_queue_full=use_qfull, enforce_queue_cap=use_cap
                    )
                    row[f"R² ({h}s)"] = r2
                    row[f"RMSE ({h}s)"] = rmse
                    row[f"MAE ({h}s)"] = mae

                results.append(row)

    return pd.DataFrame(results)

# 2. Execução das Tabelas Comparativas Discretas
df_res_discrete_departure = run_discrete_experiments(df_raw, "departure_rate_mbps")
df_res_discrete_throughput = run_discrete_experiments(df_raw, "throughput_mbps")

# Exibir resultados
print(df_res_discrete_throughput.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

##### Saída:
```
 Grau Q_Full Cap  Termos  R² (1s)  RMSE (1s)  MAE (1s)  R² (3s)  RMSE (3s)  MAE (3s)  R² (5s)  RMSE (5s)  MAE (5s)  R² (10s)  RMSE (10s)  MAE (10s)
    1    Não Não       3   0.9160     1.8587    1.6727   0.1540     4.8467    4.4698  -1.9365     7.1398    6.5568   -7.1759     10.5140     9.5520
    1    Não Sim       3   0.9160     1.8587    1.6727   0.1540     4.8467    4.4698  -1.9365     7.1398    6.5568   -7.1759     10.5140     9.5520
    1    Sim Não       3   0.9111     1.9130    1.7340   0.0198     5.2170    4.7996  -2.6552     7.9656    7.2333  -10.7021     12.5785    11.3518
    1    Sim Sim       3   0.9111     1.9130    1.7340   0.0198     5.2170    4.7996  -2.6552     7.9656    7.2333  -10.7021     12.5785    11.3518
    2    Não Não       6   0.9818     0.8647    0.6230   0.8810     1.8179    1.3264   0.6710     2.3898    1.7912    0.2997      3.0771     2.7107
    2    Não Sim       6   0.9822     0.8569    0.6166   0.8837     1.7973    1.3168   0.6804     2.3554    1.7767    0.3144      3.0447     2.6931
    2    Sim Não      11   0.9848     0.7909    0.5776   0.9025     1.6450    1.2491   0.7485     2.0893    1.6316    0.4971      2.6076     2.3324
    2    Sim Sim      11   0.9848     0.7909    0.5776   0.9025     1.6450    1.2491   0.7485     2.0893    1.6316    0.4971      2.6076     2.3324
    3    Não Não       7   0.9834     0.8274    0.6651   0.8643     1.9411    1.4643   0.5912     2.6638    1.7472    0.2917      3.0946     2.1402
    3    Não Sim       7   0.9834     0.8274    0.6651   0.8643     1.9411    1.4643   0.5912     2.6638    1.7472    0.2917      3.0946     2.1402
    3    Sim Não      16   0.9781     0.9489    0.7681   0.9154     1.5326    1.1699   0.8373     1.6807    1.2217    0.9143      1.0768     0.6712
    3    Sim Sim      16   0.9781     0.9489    0.7681   0.9154     1.5326    1.1699   0.8373     1.6807    1.2217    0.9143      1.0768     0.6712

### Observações:

- Limiar muito permissivo (0.0001);
- Teste com limiar em 0.001 no modelo discreto reduziu as variáveis para 8 (ainda com o problema do "queue_full") e com R^2 em ~0.90 para grau=3;
- Complexidade aumentar, porém o modelo considera "queue_full" como uma variável diferentes em cada grau de polinômio. O que pode ser simplificado pois é uma variável binária; [1]
- É viável e recomendado normalizar o modelo para que todas as variáveis possuam os mesmo valores de variação, como entre [0, 1]?

[1] - Exemplo: 0^3 = 0 e 1^3 = 1. Nesse caso a potência não importa, podendo reduzir o número de variáveis em 2.